# 04: Get Remediation

You have a CVE. Now find out what the fix is so you can write a ticket for the patching team.

Spotlight stores remediation records separately from vulnerability findings. A finding carries a
pointer (`remediation_id`) that you resolve into the actual fix text.

In [ ]:
import os

from falconpy import SpotlightVulnerabilities
from rich import print as rprint
from rich.table import Table

client_id = os.environ.get("FALCON_CLIENT_ID", "")
client_secret = os.environ.get("FALCON_CLIENT_SECRET", "")

vulns = SpotlightVulnerabilities(client_id=client_id, client_secret=client_secret)


def show(title, fields):
    table = Table(title=title)
    table.add_column("field", style="cyan")
    table.add_column("value")
    for k, v in fields.items():
        table.add_row(str(k), str(v))
    rprint(table)

## Explore the endpoint

A vulnerability finding does not carry its fix directly. It carries a pointer in
`apps[].remediation_info.recommended_id`. You take that ID and pass it to `get_remediations_v2`
to get the actual fix text.

Run the cell below to see how a few findings map to their remediation IDs.

In [ ]:
sample = vulns.query_vulnerabilities_combined(
    limit=3, filter="status:'open'+cve.is_cisa_kev:true", facet=["cve", "remediation"]
)

table = Table(title="A finding points at a remediation")
table.add_column("cve", style="cyan")
table.add_column("app")
table.add_column("remediation_id")
for vuln in sample["body"]["resources"]:
    app = vuln["apps"][0]
    table.add_row(vuln["cve"]["id"], app["product_name_version"],
                  app["remediation_info"]["recommended_id"])
rprint(table)

In [ ]:
# TODO: paste the cve_id from notebook 02 or 03
cve_id = None

show("Resolving remediation for", {"cve": cve_id})

## Get the remediation ID for the case CVE

The cell below queries one open finding for your CVE and pulls the `remediation_id` from it.
This is the pointer you will resolve in the next step.

In [ ]:
if not cve_id:
    print("paste the cve_id in the cell above first")
else:
    rec = vulns.query_vulnerabilities_combined(
        limit=1, filter=f"cve.id:'{cve_id}'+status:'open'", facet=["cve", "remediation"]
    )["body"]["resources"][0]

    app = rec["apps"][0]
    remediation_id = app["remediation_info"]["recommended_id"]

    show("Vulnerable app", {"app": app["product_name_version"], "remediation_id": remediation_id})

## Resolve it

Now take the `remediation_id` and pass it to `get_remediations_v2`. This returns the fix as
plain text: a `title` and an `action` (the specific version to update to).

Fill in the `ids` parameter with the `remediation_id` you just got.

In [ ]:
# TODO: resolve the remediation id into the fix text
# hint: vulns.get_remediations_v2(ids=[remediation_id])
resp = None  # replace this line with your call

if not resp or resp["status_code"] != 200:
    print("query failed or not filled in yet")
else:
    rem = resp["body"]["resources"][0]
    show("Remediation", {"title": rem["title"], "action": rem["action"]})

## The ticket line

The `action` field is what goes in a patching ticket. It names the exact package and version to
update to.

In [ ]:
if not resp or resp["status_code"] != 200:
    print("resolve the remediation in the cell above first")
else:
    fix_action = rem["action"]
    show("Fix", {"CVE": cve_id, "Fix": fix_action})

## Fallback

Run a cell here only if the query above returned nothing (tenant or seed issue). Skip it if you
already have a real value.

In [ ]:
import json

try:
    with open("data/sample_responses/remediations_query.json") as handle:
        response = {"status_code": 200, "body": json.load(handle)}
except FileNotFoundError:
    print("sample file not found")
    response = None

if response:
    rem = response["body"]["resources"][0]
    fix_action = rem["action"]
    show("Fallback", {"title": rem["title"], "action": fix_action})

## Carry to 05

Keep `cve_id` and `fix_action` for the incident summary.